# Passo Firme — treino do detector de barreiras de calçada (YOLO26)

Disciplina **Applied Computer Vision** · Global Solution 2026.2 · CitizenTech

Este notebook treina um YOLO26 sob medida para achar `buraco`, `desnivel` e `obstaculo` (e, se houver exemplos suficientes, `veiculo_na_calcada`) em fotos de calçadas de São Paulo, avalia o resultado e gera as figuras do relatório.

**Antes de rodar**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução →* **GPU T4**.
2. Clique no ícone de chave (**Secrets**) na barra lateral, crie `ROBOFLOW_API_KEY` com a chave do Roboflow (*Settings → API Keys*) e libere o acesso deste notebook. A chave nunca fica escrita no código.
3. Preencha `WORKSPACE`, `PROJETO` e `VERSAO` na seção 1.

**Ao terminar:** *Arquivo → Salvar uma cópia no GitHub* (pasta `notebooks/` do repositório), com as saídas. O professor precisa ver os resultados sem rodar nada.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q "ultralytics>=8.4.0" roboflow
import ultralytics
ultralytics.checks()

## 1. Baixar o dataset do Roboflow

A versão usada aqui junta as imagens de calçadas de São Paulo baixadas do Mapillary (arquivos `sp_<bairro>_<id>.jpg`, anotadas pelo grupo) e as 105 imagens clonadas do dataset aberto *Sidewalk obstacles* (Blindspot), com as classes de origem reagrupadas em `obstaculo` pelo *Modify Classes*. O teste tem só imagens de SP. O formato `yolov8` é o mesmo `.txt` que o YOLO26 lê.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

WORKSPACE = "seu-workspace"        # aparece na URL: app.roboflow.com/<workspace>/<projeto>
PROJETO = "passo-firme-calcadas"   # nome do projeto no Roboflow
VERSAO = 1                         # número da versão gerada em Versions (imagens de SP + Blindspot)

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJETO).version(VERSAO).download("yolov8", location="/content/dataset", overwrite=True)

In [ ]:
import yaml
from pathlib import Path

def preparar_yaml(pasta):
    """O data.yaml exportado às vezes traz caminhos relativos que quebram no Colab: fixamos os absolutos."""
    arquivo = Path(pasta) / "data.yaml"
    cfg = yaml.safe_load(arquivo.read_text())
    cfg["path"] = str(pasta)
    cfg["train"], cfg["val"] = "train/images", "valid/images"
    if (Path(pasta) / "test" / "images").exists():
        cfg["test"] = "test/images"
    else:
        cfg.pop("test", None)
    arquivo.write_text(yaml.safe_dump(cfg, allow_unicode=True))
    return str(arquivo), cfg

DATA, cfg = preparar_yaml(dataset.location)
SPLIT = "test" if "test" in cfg else "val"          # conjunto usado na avaliação final
PASTA_TESTE = "test" if SPLIT == "test" else "valid"
NOMES = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
print("Classes:", NOMES, "| avaliação final em:", SPLIT)

In [ ]:
import pandas as pd
from collections import Counter

linhas = []
for rotulo_conjunto, pasta in [("treino", "train"), ("validação", "valid"), ("teste", "test")]:
    base = Path(dataset.location) / pasta
    if not base.exists():
        continue
    contagem, nulas = Counter(), 0
    imagens = [p for p in (base / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    for img in imagens:
        rot = base / "labels" / (img.stem + ".txt")
        anot = [l for l in rot.read_text().splitlines() if l.strip()] if rot.exists() else []
        nulas += not anot
        for l in anot:
            contagem[int(float(l.split()[0]))] += 1
    linhas.append({"conjunto": rotulo_conjunto, "imagens": len(imagens), "sem barreira": nulas,
                   **{n: contagem.get(i, 0) for i, n in enumerate(NOMES)}})

tabela_dataset = pd.DataFrame(linhas)
tabela_dataset   # vai para o PDF (seção Metodologia)

## 2. Treino-relâmpago (só para testar o caminho)

Dez épocas confirmam que dados, classes e caminhos estão certos antes de gastar tempo de GPU. **Não use estes números no relatório.**

In [ ]:
from ultralytics import YOLO

YOLO("yolo26n.pt").train(data=DATA, epochs=10, imgsz=640, batch=16,
                         project="/content/runs", name="00_relampago", exist_ok=True)

## 3. Experimento A — YOLO26n (modelo principal)

Ajuste fino (*transfer learning*) a partir dos pesos pré-treinados no COCO. Hiperparâmetros para registrar no PDF:

| Parâmetro | Valor |
| --- | --- |
| Modelo base | `yolo26n.pt` (COCO) |
| Tamanho da imagem | 640 |
| Épocas | até 80, com parada antecipada após 20 sem melhora (`patience`) |
| Lote | 16 |
| Semente | 42 |
| Aumento de dados | padrão do Ultralytics: mosaico, variação de cor (HSV), escala, translação e espelhamento horizontal. Sem espelhamento vertical: a calçada nunca aparece de cabeça para baixo. |

In [ ]:
EPOCAS = 80
comum = dict(data=DATA, imgsz=640, epochs=EPOCAS, patience=20, batch=16, seed=42,
             project="/content/runs", exist_ok=True, plots=True)

YOLO("yolo26n.pt").train(name="A_yolo26n", **comum)
A_BEST = "/content/runs/A_yolo26n/weights/best.pt"

In [ ]:
from IPython.display import Image, display
display(Image("/content/runs/A_yolo26n/results.png", width=950))   # curvas de perda e métricas por época

## 4. Avaliação no conjunto de teste

O teste tem fotos que o modelo nunca viu, de ruas separadas antes do treino. Métricas:
- **Precisão:** das caixas que o modelo desenhou, quantas estavam certas.
- **Recall:** das barreiras que existiam, quantas o modelo achou. Para quem anda com bengala ou cadeira de rodas, um buraco não detectado custa mais que um falso alarme.
- **mAP50:** média, entre as classes, da área sob a curva precisão × recall, contando como acerto a caixa com IoU ≥ 0,5.
- **mAP50-95:** a mesma média, exigindo caixas cada vez mais justas (IoU de 0,5 a 0,95).

In [ ]:
def avaliar(pesos, nome):
    m = YOLO(pesos).val(data=DATA, split=SPLIT, imgsz=640, batch=16,
                        project="/content/runs", name=nome, exist_ok=True)
    por_classe = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        por_classe.append({"classe": m.names[int(c)], "precisão": p, "recall": r, "mAP50": ap50, "mAP50-95": ap})
    geral = {"precisão": m.box.mp, "recall": m.box.mr, "mAP50": m.box.map50, "mAP50-95": m.box.map,
             "ms por imagem (GPU T4)": m.speed["inference"]}
    return pd.DataFrame(por_classe).round(3), geral

tabelaA, geralA = avaliar(A_BEST, "A_teste")
print({k: round(v, 3) for k, v in geralA.items()})
tabelaA   # tabela por classe: vai para o PDF (Resultados)

In [ ]:
import glob
for f in sorted(glob.glob("/content/runs/A_teste/*.png")):   # matriz de confusão, curvas PR e F1
    print(Path(f).name)
    display(Image(f, width=700))

## 5. Experimento B — YOLO26s (comparação de tamanho)

Mesmo dataset e mesmos hiperparâmetros, modelo maior. Mostra no PDF a troca entre precisão e velocidade e justifica a escolha do nano para o tempo real.

In [ ]:
YOLO("yolo26s.pt").train(name="B_yolo26s", **comum)
B_BEST = "/content/runs/B_yolo26s/weights/best.pt"
tabelaB, geralB = avaliar(B_BEST, "B_teste")

comparacao = pd.DataFrame([{"modelo": "YOLO26n", **geralA}, {"modelo": "YOLO26s", **geralB}]).round(3)
comparacao   # vai para o PDF

## 6. Desempenho por grupo de imagens (IA responsável)

O detector acerta igual em todo tipo de calçada? O grupo sai do nome do arquivo: `sp_<bairro>_<id>.jpg` vira o bairro (o Roboflow mantém esse começo do nome) e o resto entra como "dataset aberto". `REGIOES` junta bairros em regiões; bairro fora dela aparece com o próprio nome.

- **Teste** (só SP): compara os bairros ou regiões presentes. Se entrarem fotos da periferia, nomeie como `sp_periferia_<n>.jpg`.
- **Validação**: compara SP com o dataset aberto.

Grupos com menos de ~30 imagens dão números instáveis: diga isso no relatório.

In [ ]:
import tempfile

REGIOES = {"se": "centro", "paulista": "centro", "itaquera": "periferia", "jabaquara": "periferia", "periferia": "periferia"}

def grupo_da_imagem(nome):
    partes = nome.lower().split("_")
    if partes[0] == "sp" and len(partes) > 2:
        return "SP: " + REGIOES.get(partes[1], partes[1])
    return "dataset aberto"

def avaliar_grupos(pesos, pasta_split, so_sp=False):
    imagens = sorted((Path(dataset.location) / pasta_split / "images").glob("*"))
    separadas = {}
    for img in imagens:
        g = "SP (todas)" if so_sp and img.name.lower().startswith("sp_") else grupo_da_imagem(img.name)
        separadas.setdefault(g, []).append(img)
    modelo, linhas = YOLO(pesos), []
    for grupo, lista in sorted(separadas.items()):
        pasta = Path(tempfile.mkdtemp())
        (pasta / "lista.txt").write_text("\n".join(map(str, lista)))
        (pasta / "grupo.yaml").write_text(yaml.safe_dump(
            {"path": dataset.location, "train": str(pasta / "lista.txt"), "val": str(pasta / "lista.txt"), "names": cfg["names"]},
            allow_unicode=True))
        m = modelo.val(data=str(pasta / "grupo.yaml"), split="val", imgsz=640, plots=False, verbose=False)
        linhas.append({"conjunto": pasta_split, "grupo": grupo, "imagens": len(lista), "precisão": m.box.mp,
                       "recall": m.box.mr, "mAP50": m.box.map50, "mAP50-95": m.box.map})
    return pd.DataFrame(linhas).round(3)

por_grupo = pd.concat([avaliar_grupos(A_BEST, PASTA_TESTE), avaliar_grupos(A_BEST, "valid", so_sp=True)], ignore_index=True)
por_grupo   # vai para o PDF

## 7. Exemplos de resultados e análise de erros

Compara, imagem a imagem do teste, as caixas anotadas (verde) com as previstas (vermelho). Acerto = mesma classe e IoU ≥ 0,5. Os piores casos alimentam a seção de limitações do PDF; os melhores viram as figuras de exemplo.

In [ ]:
import cv2
import matplotlib.pyplot as plt

melhor = YOLO(A_BEST)
pasta_img = Path(dataset.location) / PASTA_TESTE / "images"
pasta_lbl = Path(dataset.location) / PASTA_TESTE / "labels"

def caixas_reais(img_path, w, h):
    f = pasta_lbl / (img_path.stem + ".txt")
    caixas = []
    if f.exists():
        for l in f.read_text().splitlines():
            if l.strip():
                c, xc, yc, bw, bh = map(float, l.split()[:5])
                caixas.append((int(c), (xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h))
    return caixas

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    uniao = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / uniao if uniao > 0 else 0

casos = []
for p in sorted(pasta_img.glob("*")):
    r = melhor.predict(str(p), conf=0.35, imgsz=640, verbose=False)[0]
    h, w = r.orig_shape
    reais = caixas_reais(p, w, h)
    previstas = [(int(c), *xyxy) for xyxy, c in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist())]
    usadas = set()
    for real in reais:
        for j, prev in enumerate(previstas):
            if j not in usadas and prev[0] == real[0] and iou(real[1:], prev[1:]) >= 0.5:
                usadas.add(j)
                break
    casos.append({"imagem": p.name, "anotadas": len(reais), "previstas": len(previstas), "acertos": len(usadas),
                  "falsos negativos": len(reais) - len(usadas), "falsos positivos": len(previstas) - len(usadas),
                  "_r": r, "_reais": reais})

def painel(lista, titulo, arquivo):
    fig, eixos = plt.subplots(2, 3, figsize=(18, 10))
    for ax in eixos.flat:
        ax.axis("off")
    for ax, caso in zip(eixos.flat, lista):
        img = caso["_r"].orig_img.copy()
        for c, x1, y1, x2, y2 in caso["_reais"]:
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 4)
        for xyxy, c, cf in zip(caso["_r"].boxes.xyxy.tolist(), caso["_r"].boxes.cls.tolist(), caso["_r"].boxes.conf.tolist()):
            x1, y1, x2, y2 = map(int, xyxy)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 3)
            cv2.putText(img, f"{caso['_r'].names[int(c)]} {cf:.2f}", (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"FN={caso['falsos negativos']}  FP={caso['falsos positivos']}  acertos={caso['acertos']}", fontsize=11)
    fig.suptitle(titulo + "  (verde = anotação, vermelho = previsão)", fontsize=14)
    plt.tight_layout()
    plt.savefig(arquivo, dpi=110)
    plt.show()

bons = sorted(casos, key=lambda c: (c["acertos"], -c["falsos positivos"]), reverse=True)[:6]
piores = sorted(casos, key=lambda c: c["falsos negativos"] + c["falsos positivos"], reverse=True)[:6]
painel(bons, "Exemplos de acerto", "/content/runs/exemplos_acerto.png")
painel(piores, "Piores casos", "/content/runs/piores_casos.png")

resumo_erros = pd.DataFrame([{k: v for k, v in c.items() if not k.startswith("_")} for c in casos])
resumo_erros.sum(numeric_only=True)

Para o PDF, escolham **5 erros** dos piores casos e escrevam, para cada um: o que o modelo fez, a causa provável (sombra, folhas cobrindo o buraco, ângulo, classe confundida, anotação inconsistente) e a correção proposta.

## 8. Velocidade em CPU

O tempo real roda no notebook de vocês, sem GPU. Esta medida no CPU do Colab dá uma ideia; a medida que vai para o PDF sai do `src/tempo_real.py` rodando na máquina de vocês.

In [ ]:
import time

modelo_cpu = YOLO(A_BEST)
amostra = sorted(pasta_img.glob("*"))[:30]
modelo_cpu.predict(str(amostra[0]), device="cpu", verbose=False)   # aquecimento
inicio = time.perf_counter()
for p in amostra:
    modelo_cpu.predict(str(p), device="cpu", imgsz=640, verbose=False)
ms = (time.perf_counter() - inicio) / len(amostra) * 1000
print(f"CPU do Colab: {ms:.0f} ms por imagem, cerca de {1000 / ms:.1f} FPS")

## 9. Salvar os resultados

Copia tudo para o Google Drive e baixa o `best.pt`, que vai para a pasta `modelos/` do repositório (o arquivo do nano tem poucos MB e cabe no GitHub).

In [ ]:
import shutil
from google.colab import drive, files

drive.mount("/content/drive")
destino = "/content/drive/MyDrive/passo-firme-visao"
shutil.copytree("/content/runs", f"{destino}/runs", dirs_exist_ok=True)
for nome, tabela in [("dataset", tabela_dataset), ("teste_por_classe_A", tabelaA), ("comparacao_n_x_s", comparacao), ("por_grupo", por_grupo)]:
    tabela.to_csv(f"{destino}/{nome}.csv", index=False)

shutil.copy(A_BEST, "/content/best.pt")
files.download("/content/best.pt")

## O que deste notebook vai para o PDF

| Seção do PDF | O que levar |
| --- | --- |
| Metodologia | `tabela_dataset`, tabela de hiperparâmetros (seção 3), `results.png` |
| Resultados | `tabelaA` (por classe), `comparacao` (n × s), matriz de confusão normalizada, `exemplos_acerto.png` |
| IA responsável | `por_grupo` (bairros/regiões no teste; SP × dataset aberto na validação) |
| Limitações | `piores_casos.png` e os 5 erros comentados |